## Databricks IP Functions (Beta - DBR 18.2+)

Databricks introduced a set of **IP functions** that operate on IPv4 and IPv6 addresses and CIDR blocks represented as `STRING` or `BINARY` values.

| Function | Description |
| --- | --- |
| `ip_host` | Validate & canonicalize an IP address |
| `ip_network` | Get network portion of a CIDR block |
| `ip_network_last` | Get last (broadcast) address of a CIDR block |
| `ip_prefix_length` | Get prefix length from a CIDR block |
| `ip_version` | Return IP version (4 or 6) |
| `ip_cidr` | Canonicalize a CIDR block |
| `ip_cidr_contains` | Check if an IP/CIDR is within another CIDR |
| `ip_as_binary` / `ip_as_string` | Convert between binary and string |
| `try_*` variants | Safe versions that return NULL instead of error |

In [0]:
-- ip_host: validates and returns canonical form of an IP address
SELECT
  ip_host('192.168.1.5')                              AS ipv4_canonical,
  ip_host('2001:0DB8:0000:0000:0000:0000:0000:0001') AS ipv6_canonical,
  ip_host('::ffff:192.0.2.128')                      AS ipv4_mapped_ipv6

In [0]:
-- ip_version: returns 4 or 6 depending on address type
SELECT
  ip_version('10.0.0.1')       AS v4_result,
  ip_version('2001:db8::1')    AS v6_result,
  ip_version('::ffff:10.0.0.1') AS mapped_result

In [0]:
-- ip_network: first address (network address) of CIDR block
-- ip_network_last: last address (broadcast) of CIDR block
SELECT
  '192.168.1.130/24'                          AS cidr_block,
  ip_network('192.168.1.130/24')              AS network_first,
  ip_network_last('192.168.1.130/24')         AS network_last,
  ip_prefix_length('192.168.1.130/24')        AS prefix_len
UNION ALL
SELECT
  '10.0.0.50/16',
  ip_network('10.0.0.50/16'),
  ip_network_last('10.0.0.50/16'),
  ip_prefix_length('10.0.0.50/16')
UNION ALL
SELECT
  '2001:db8::1/32',
  ip_network('2001:db8::1/32'),
  ip_network_last('2001:db8::1/32'),
  ip_prefix_length('2001:db8::1/32')

In [0]:
-- ip_cidr_contains: check if an IP or CIDR is contained within another CIDR
SELECT
  ip_cidr_contains('192.168.1.0/24', '192.168.1.100')    AS ip_in_subnet,
  ip_cidr_contains('192.168.1.0/24', '192.168.2.1')      AS ip_not_in_subnet,
  ip_cidr_contains('10.0.0.0/8', '10.255.255.255')       AS ip_in_class_a,
  ip_cidr_contains('10.0.0.0/8', '10.0.0.0/16')          AS subnet_in_supernet

In [0]:
-- ip_cidr: returns canonical CIDR representation (normalizes host bits)
SELECT
  ip_cidr('192.168.1.130/24')                              AS normalized_v4,
  ip_cidr('2001:0db8:0000:0000:0000:0000:0000:0001/32')   AS normalized_v6

In [0]:
-- try_ip_host: returns NULL for invalid IPs instead of raising an error
SELECT
  try_ip_host('192.168.1.1')   AS valid_ip,
  try_ip_host('999.999.999.0') AS invalid_ip,
  try_ip_host('not_an_ip')     AS garbage_input,
  try_ip_host(NULL)            AS null_input

In [0]:
-- Practical example: classify web traffic IPs into network zones
WITH access_logs AS (
  SELECT * FROM VALUES
    ('192.168.1.45',  'GET /api/users'),
    ('10.0.5.22',     'POST /api/orders'),
    ('172.16.0.100',  'GET /dashboard'),
    ('8.8.8.8',       'GET /public'),
    ('2001:db8::42',  'GET /api/v2/data')
  AS t(client_ip, request)
)
SELECT
  client_ip,
  request,
  ip_version(client_ip) AS ip_ver,
  CASE
    WHEN ip_cidr_contains('192.168.0.0/16', client_ip) THEN 'Private (Class C)'
    WHEN ip_cidr_contains('10.0.0.0/8', client_ip)     THEN 'Private (Class A)'
    WHEN ip_cidr_contains('172.16.0.0/12', client_ip)  THEN 'Private (Class B)'
    ELSE 'Public / External'
  END AS network_zone
FROM access_logs

In [0]:
-- Every IP function demonstrated in a single SELECT
SELECT
  ip_host('2001:0DB8:0000:0000:0000:0000:0000:0001')    AS ip_host,

  ip_version('192.168.1.5')                             AS ip_version,

  ip_network('192.168.1.5/24')                          AS ip_network,

  ip_network_first('192.168.1.5/24')                    AS ip_network_first,

  ip_network_last('192.168.1.5/24')                     AS ip_network_last,

  ip_prefix_length('192.168.1.5/24')                    AS ip_prefix_length,

  ip_cidr('192.168.1.5/24')                             AS ip_cidr,

  ip_cidr_contains('192.168.1.0/24', '192.168.1.5')    AS ip_cidr_contains,

  ip_as_binary('192.168.1.5')                           AS ip_as_binary,

  ip_as_string(ip_as_binary('192.168.1.5'))             AS ip_as_string,

  try_ip_host('invalid')                                AS try_ip_host,

  try_ip_cidr('invalid/99')                             AS try_ip_cidr,

  try_ip_as_binary('invalid')                           AS try_ip_as_binary,

  try_ip_as_string(X'00')                               AS try_ip_as_string
  